# 🌌 Exploratory Data Analysis — MaNGA galaxy morphology metadata & integrated spectra

This notebook performs a **general EDA** of two linked datasets stored in Google Drive:

| Dataset | Format | Content | Columns |
|---|---|---|---|
| **Metadata** | `morfologia.fits` (~16k rows) | one row per galaxy | `name, objra, objdec, Type, Ttype, nsa_z, LogMass, incl, PETRO_TH90` |
| **Spectra** | ~16k `*.fits.gz` files inside one `.zip` | one file per galaxy, named after `name` (e.g. `manga-10001-1901.fits.gz`) | `WAVE, FLUX_1RE, FLUX_FoV, ERROR_1RE, ERROR_FoV` |

**Sections**
1. Setup & configuration
2. Metadata EDA (structure, missing values, morphology, distributions, correlations, sky, outliers)
3. Spectra files: unzip, inventory, cross-match with metadata, anatomy of one file
4. Bulk loading of all spectra (parallel + cache) and spectral quality-control EDA
5. Spectral indices (D4000, emission/absorption equivalent widths, colour)
6. **Early- vs late-type galaxies**: how metadata and spectra relate to morphology, and how to plot spectra grouped by `Type`

**How to run:** edit the paths in §1.2 → `Runtime ▸ Run all`. The first run reads all 16k spectra (several minutes) and writes a cache to Drive; later runs load the cache in seconds.

> **Assumptions (check them in §3.3):**
> * `WAVE` is in Å and in the **observed frame** (all sample files start at 3622 Å, MaNGA's blue limit, regardless of redshift), so spectra are de-redshifted with `nsa_z`. Set `WAVE_IS_OBSERVED = False` if your files are already rest-frame.
> * `FLUX_1RE` / `FLUX_FoV` are spectra integrated within 1 effective radius and over the full IFU field of view; `ERROR_*` are their 1σ uncertainties.
> * `-9999` is a "no data" sentinel (e.g. `nsa_z` of some galaxies).

## 1. Setup
### 1.1 Install & import libraries
* **astropy** – FITS I/O (`astropy.io.fits`, `astropy.table.Table`), units, coordinates and cosmology.
* **pandas / numpy / seaborn / matplotlib** – tabular EDA and plotting.
* **joblib + tqdm** – parallel reading of 16k files with a progress bar.
* **scikit-learn / scipy** – PCA, simple classifiers and statistical tests (section 6).
* *Optional* (section 6.8): **sdss-marvin** (MaNGA images/maps), **astroNN** (deep-learning models).

In [ ]:
# astropy is usually pre-installed on Colab; --upgrade makes sure we have a recent version
!pip install -q --upgrade astropy

In [ ]:
# --- Standard library ---------------------------------------------------------
import shutil, zipfile, warnings
from pathlib import Path

# --- Scientific stack ---------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from tqdm.auto import tqdm
from joblib import Parallel, delayed

# --- Astronomy ----------------------------------------------------------------
from astropy.io import fits                 # low-level FITS access (HDUs, headers)
from astropy.table import Table             # high-level table reader (handles .fits.gz)
from astropy import units as u              # physical units
from astropy.coordinates import SkyCoord    # sky coordinates
from astropy.cosmology import Planck18 as cosmo  # angular scale -> physical size

# --- Display settings ---------------------------------------------------------
warnings.filterwarnings('ignore', category=RuntimeWarning)  # e.g. nanmedian of all-NaN columns
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({'figure.figsize': (11, 4.5), 'figure.dpi': 100})
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4g}'.format)

# One colormap for the Hubble sequence, used everywhere: dark red (E) -> green -> dark blue (late spirals)
from matplotlib.colors import LinearSegmentedColormap
HUBBLE_CMAP = LinearSegmentedColormap.from_list('hubble', ['#8b0000', '#e66101', '#1b9e77', '#2c7bb6', '#253494'])

### 1.2 Mount Google Drive and configure paths
Edit the three `# <-- EDIT` lines to match your Drive layout.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Input locations (Google Drive) -----------------------------------------
DRIVE_DIR  = Path('/content/drive/MyDrive/manga')   # <-- EDIT: folder holding the data
META_FILE  = DRIVE_DIR / 'morfologia.fits'          # <-- EDIT: metadata FITS table
ZIP_FILE   = DRIVE_DIR / 'manga_spectra.zip'        # <-- EDIT: zip with the 16k *.fits.gz

# ---- Working locations -------------------------------------------------------
LOCAL_ZIP  = Path('/content') / ZIP_FILE.name       # local copy: one big read from Drive is much faster
LOCAL_DIR  = Path('/content/spectra')               # unzipped spectra on the fast local disk
CACHE_NPZ  = DRIVE_DIR / 'eda_spectra_cache.npz'    # resampled spectra (persisted between sessions)
CACHE_CSV  = DRIVE_DIR / 'eda_spectra_stats.csv'    # per-spectrum QC statistics

# ---- Analysis options ----------------------------------------------------------
N_MAX            = None    # e.g. 500 for a quick test run; None = all spectra (delete cache when changing)
WAVE_IS_OBSERVED = True    # True -> WAVE is observed-frame and gets divided by (1 + nsa_z)
REST_GRID        = np.arange(3300., 10000., 2.0)   # common rest-frame grid [Å] for all spectra
NORM_WINDOW      = (5400., 5600.)                  # quiet continuum window used to normalise spectra
SNR_MIN          = 5       # minimum median S/N (FLUX_1RE) for a spectrum to be considered "good"
SEED             = 42
rng = np.random.default_rng(SEED)

# ---- Column names --------------------------------------------------------------
WAVE_COL  = 'WAVE'
FLUX_COLS = ['FLUX_1RE', 'FLUX_FoV']
ERR_COLS  = ['ERROR_1RE', 'ERROR_FoV']
NUM_COLS  = ['nsa_z', 'LogMass', 'incl', 'PETRO_TH90']   # numeric metadata columns

print('Metadata exists:', META_FILE.exists(), '| Zip exists:', ZIP_FILE.exists())

## 2. Metadata EDA (`morfologia.fits`)
### 2.1 Load the FITS table
`fits.info` lists the HDUs; `Table.read` loads the first table HDU, and `.to_pandas()` gives a DataFrame.

In [ ]:
# List HDUs (extensions) in the file: index, name, type, dimensions
fits.info(META_FILE)

# Read the first table HDU (astropy keeps units / masks from the FITS header)
meta_tab = Table.read(META_FILE)
for col in meta_tab.columns.values():                 # name, dtype and unit of every column
    print(f'{col.name:12s} {str(col.dtype):8s} unit={col.unit}')

# Convert to pandas (masked values become NaN)
meta = meta_tab.to_pandas()

# FITS stores strings as fixed-width bytes -> decode and strip trailing spaces
for col in meta.columns:
    if meta[col].dtype == object:
        meta[col] = meta[col].apply(lambda v: v.decode().strip() if isinstance(v, bytes) else v)

# -9999 is the NSA/SDSS "no data" sentinel -> NaN
meta = meta.replace(-9999, np.nan)

print(f'\n{len(meta):,} galaxies x {meta.shape[1]} columns')
meta.head()

### 2.2 Structure, duplicates and missing values

In [ ]:
meta.info()   # dtypes, non-null counts, memory

# Duplicated names would break the metadata <-> spectrum join (1 file per name)
dups = meta['name'].duplicated(keep=False)
print('\nDuplicated names:', dups.sum())
meta = meta.drop_duplicates('name').reset_index(drop=True)

In [ ]:
# Percentage of missing values per column
missing = meta.isna().mean().mul(100).sort_values(ascending=False)

ax = missing.plot.bar(color='tab:red', figsize=(9, 3))
ax.set(ylabel='% missing', title='Missing values per column (after -9999 -> NaN)')
plt.show()

# Rows that have a Type but no redshift cannot be de-redshifted later
print('Galaxies without nsa_z :', meta['nsa_z'].isna().sum())
print('Galaxies with incl == 0:', (meta['incl'] == 0).sum(), '(face-on or a default value? worth checking)')

### 2.3 Descriptive statistics

In [ ]:
# Extended percentiles help spot extreme tails / unphysical values
meta[NUM_COLS + ['Ttype']].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

### 2.4 Morphology columns (`Type`, `Ttype`)
`Type` is a Hubble-type string (e.g. `E`, `S0`, `SABbc`, `SBab`) and `Ttype` its numerical stage
(E = −5, S0 = −2, S0a = 0, Sa = 1 … Sd = 7; `S` = 11 seems to be "spiral, sub-type unknown").

We derive helper columns:
* `hubble_class` – `Type` without the bar code (`SABbc` → `Sbc`, `Edc` → `E`)
* `bar` – `strong` (`SB`), `weak` (`SAB`) or `none`
* `morph_group` – `E` (T ≤ −3), `S0/S0a` (−3 < T ≤ 0), `Spiral` (T > 0)
* `early_late` – **Early** (T ≤ 0: E, S0, S0a) vs **Late** (T > 0: spirals). Change `EARLY_MAX_T` if you prefer S0a as late type.

In [ ]:
EARLY_MAX_T = 0   # Ttype <= EARLY_MAX_T -> early type

def hubble_class(t):
    # Strip the bar code so that e.g. 'SABbc' and 'SBbc' both become 'Sbc'
    t = str(t).strip()
    if t.startswith('E'):
        return 'E'                       # 'E', 'Edc' ... -> 'E'
    for prefix in ('SAB', 'SB', 'SA'):   # check 'SAB' before 'SB'/'SA'
        if t.startswith(prefix):
            return 'S' + t[len(prefix):]
    return t

def bar_code(t):
    t = str(t)
    return 'weak' if t.startswith('SAB') else 'strong' if t.startswith('SB') else 'none'

meta['Type']         = meta['Type'].fillna('Unknown')
meta['hubble_class'] = meta['Type'].apply(hubble_class)
meta['bar']          = meta['Type'].apply(bar_code)
meta['morph_group']  = pd.cut(meta['Ttype'], [-np.inf, -3, 0, np.inf], labels=['E', 'S0/S0a', 'Spiral'])
meta['early_late']   = np.where(meta['Ttype'].isna(), None,
                                np.where(meta['Ttype'] <= EARLY_MAX_T, 'Early', 'Late'))

# Order categories along the Hubble sequence (median Ttype), used by every plot below
TYPE_ORDER   = meta.groupby('Type')['Ttype'].median().sort_values().index.tolist()
HUBBLE_ORDER = meta.groupby('hubble_class')['Ttype'].median().sort_values().index.tolist()

# Consistency check: each Type string should map to a single Ttype
n_t = meta.groupby('Type')['Ttype'].nunique()
print('Types with >1 Ttype value:', n_t[n_t > 1].to_dict() or 'none')
print(meta['early_late'].value_counts(dropna=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [3, 1]})

# Number of galaxies per Type, ordered along the Hubble sequence
sns.countplot(data=meta, x='Type', order=TYPE_ORDER, hue='early_late', dodge=False, ax=axes[0])
axes[0].tick_params(axis='x', rotation=90)
axes[0].set_title('Galaxies per Type (ordered by Ttype)')

# Bar fraction
meta['bar'].value_counts().plot.bar(ax=axes[1], color='tab:gray')
axes[1].set_title('Bar classification')
plt.tight_layout(); plt.show()

# Ttype distribution and simplified classes
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
sns.countplot(data=meta, x='Ttype', ax=axes[0], color='tab:blue').set_title('Ttype distribution')
sns.countplot(data=meta, x='hubble_class', order=HUBBLE_ORDER, ax=axes[1], color='tab:purple')
axes[1].set_title('Hubble class (bars merged)')
plt.tight_layout(); plt.show()

### 2.5 Distributions of the numerical metadata
Densities are normalised per class so early and late types can be compared despite different sample sizes.
We also derive the **physical size** `R90_kpc` from `PETRO_TH90` (arcsec) using the Planck18 cosmology.

In [ ]:
# Angular -> physical size: kpc per arcsec at each galaxy's redshift (only valid z > 0)
z_ok  = meta['nsa_z'] > 0
scale = cosmo.kpc_proper_per_arcmin(meta.loc[z_ok, 'nsa_z'].values).to(u.kpc / u.arcsec).value
meta['R90_kpc'] = np.nan
meta.loc[z_ok, 'R90_kpc'] = meta.loc[z_ok, 'PETRO_TH90'] * scale

PLOT_COLS = NUM_COLS + ['R90_kpc']
fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(20, 3.8))
for ax, col in zip(axes, PLOT_COLS):
    sns.histplot(data=meta, x=col, hue='early_late', element='step', stat='density',
                 common_norm=False, bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

### 2.6 Correlations and pairwise relations
Spearman (rank) correlation is robust to the skewed distributions above.

In [ ]:
corr = meta[PLOT_COLS + ['Ttype']].corr(method='spearman')
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', center=0, square=True)
plt.title('Spearman correlation'); plt.show()

# Pairplot on a random subsample (fast and less over-plotted)
sample = meta.dropna(subset=['early_late']).sample(min(3000, len(meta)), random_state=SEED)
sns.pairplot(sample, vars=PLOT_COLS, hue='early_late', corner=True,
             plot_kws={'s': 6, 'alpha': .4}, diag_kws={'common_norm': False})
plt.show()

In [ ]:
# Stellar mass and size along the Hubble sequence
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
sns.boxplot(data=meta, x='hubble_class', y='LogMass', order=HUBBLE_ORDER, ax=axes[0], color='tab:orange')
sns.boxplot(data=meta, x='hubble_class', y='R90_kpc', order=HUBBLE_ORDER, ax=axes[1], color='tab:green')
axes[1].set_yscale('log')
for ax in axes: ax.tick_params(axis='x', rotation=45)
plt.tight_layout(); plt.show()

# Redshift vs mass: MaNGA selection (more massive galaxies observed at higher z)
sns.scatterplot(data=meta, x='nsa_z', y='LogMass', hue='early_late', s=6, alpha=.5)
plt.xscale('log'); plt.title('Selection: redshift vs stellar mass'); plt.show()

### 2.7 Sky distribution

In [ ]:
# SkyCoord handles RA wrapping; the Aitoff projection needs radians in [-pi, pi]
coords = SkyCoord(ra=meta['objra'].values * u.deg, dec=meta['objdec'].values * u.deg)

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(111, projection='aitoff')
for label, color in [('Early', 'tab:red'), ('Late', 'tab:blue')]:
    sel = (meta['early_late'] == label).values
    ax.scatter(coords.ra.wrap_at(180 * u.deg).radian[sel], coords.dec.radian[sel],
               s=2, alpha=.4, color=color, label=label)
ax.grid(True); ax.legend(loc='lower right'); ax.set_title('Sky distribution (RA, Dec)', pad=20)
plt.show()

### 2.8 Outliers (IQR rule)
Values beyond 3×IQR from the quartiles are flagged — not necessarily wrong, but worth a look.

In [ ]:
def iqr_outliers(s, k=3.0):
    # Boolean mask of values outside [Q1 - k*IQR, Q3 + k*IQR]
    q1, q3 = s.quantile([.25, .75])
    return (s < q1 - k * (q3 - q1)) | (s > q3 + k * (q3 - q1))

outliers = pd.DataFrame({c: iqr_outliers(meta[c]) for c in PLOT_COLS})
print(outliers.sum().to_string())
meta[outliers.any(axis=1)].head(10)

## 3. Spectroscopic files
### 3.1 Copy and unzip
Reading 16k small files directly from Drive is very slow, so we copy the zip once to the Colab disk and extract it there.

In [ ]:
# 1) Copy the zip from Drive to local disk (skipped if already done in this session)
if not LOCAL_ZIP.exists():
    shutil.copy(ZIP_FILE, LOCAL_ZIP)
print(f'Zip size: {LOCAL_ZIP.stat().st_size / 1e9:.2f} GB')

# 2) Extract the *.fits.gz members (skipped if already extracted)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(LOCAL_ZIP) as zf:
    members = [m for m in zf.namelist() if m.endswith('.fits.gz')]
    print(f'{len(members):,} .fits.gz files in the zip')
    if not any(LOCAL_DIR.rglob('*.fits.gz')):
        for m in tqdm(members, desc='Unzipping'):
            zf.extract(m, LOCAL_DIR)

### 3.2 Inventory and cross-match with the metadata

In [ ]:
def name_of(path):
    # 'manga-10001-1901.fits.gz' -> 'manga-10001-1901'
    return Path(path).name.split('.fits')[0]

files = sorted(LOCAL_DIR.rglob('*.fits.gz'))         # rglob: works even if the zip has sub-folders
spec_index = pd.DataFrame({'path': [str(p) for p in files]})
spec_index['name']    = spec_index['path'].apply(name_of)
spec_index['size_kb'] = [p.stat().st_size / 1024 for p in files]

# Set comparison between spectra and metadata names
in_spec, in_meta = set(spec_index['name']), set(meta['name'])
print(f'Spectra files            : {len(in_spec):,}')
print(f'Metadata rows            : {len(in_meta):,}')
print(f'Matched (both)           : {len(in_spec & in_meta):,}')
print(f'Spectra without metadata : {len(in_spec - in_meta):,}', sorted(in_spec - in_meta)[:5])
print(f'Metadata without spectrum: {len(in_meta - in_spec):,}', sorted(in_meta - in_spec)[:5])

spec_index['size_kb'].plot.hist(bins=50, title='Compressed file size [kB]'); plt.show()

### 3.3 Anatomy of one spectrum file
Check HDUs, header keywords, units and the wavelength sampling. **Verify here whether `WAVE` is observed- or rest-frame.**

In [ ]:
example_path = spec_index['path'].iloc[0]
print(name_of(example_path))
fits.info(example_path)                       # HDU list

hdr = fits.getheader(example_path, ext=1)     # header of the table extension
print(repr(hdr)[:2000])                       # first keywords (TTYPE/TUNIT give columns & units)

spec = Table.read(example_path)               # astropy decompresses .gz transparently
print(spec.info)

wave = np.asarray(spec[WAVE_COL], float)
dw = np.diff(wave)
print(f'Pixels: {len(wave)} | range: {wave.min():.1f}-{wave.max():.1f} Å | '
      f'median step: {np.median(dw):.2f} Å | gaps (>1.5 step): {(dw > 1.5 * np.median(dw)).sum()}')
spec[:5]

In [ ]:
def break_gaps(wave, *arrays, factor=1.5):
    # Insert NaN where the wavelength jumps by > factor*step (masked pixels),
    # so matplotlib does not draw straight lines across missing regions.
    wave = np.asarray(wave, float)
    cut = np.where(np.diff(wave) > factor * np.median(np.diff(wave)))[0] + 1
    return [np.insert(np.asarray(a, float), cut, np.nan) for a in (wave, *arrays)]

def plot_raw_spectrum(path, ax=None):
    # Plot FLUX_1RE and FLUX_FoV with their 1-sigma error bands (observed frame)
    t = Table.read(path)
    ax = ax or plt.gca()
    for f, e, c in zip(FLUX_COLS, ERR_COLS, ['tab:blue', 'tab:orange']):
        w, fl, er = break_gaps(t[WAVE_COL], t[f], t[e])
        ax.plot(w, fl, lw=.6, color=c, label=f)
        ax.fill_between(w, fl - er, fl + er, color=c, alpha=.2)
    row = meta.set_index('name').loc[name_of(path)] if name_of(path) in in_meta else None
    title = name_of(path) + ('' if row is None else f"  |  Type={row['Type']}  z={row['nsa_z']:.3f}")
    ax.set(title=title, xlabel='WAVE [Å]', ylabel='flux')
    ax.legend()
    return ax

plt.figure(figsize=(14, 4)); plot_raw_spectrum(example_path); plt.show()

> **Tip:** in the sample CSVs, `ERROR_FoV` is almost constant and often *larger* than `FLUX_FoV`, while `ERROR_1RE` behaves like a normal 1σ error. Check whether `ERROR_FoV` is really a 1σ error (or e.g. a variance or a summed quantity) before using it for weighting. The S/N statistics in §4.2 will show this for the whole sample.

## 4. Bulk loading of all spectra
For each file we (i) compute quality statistics in the native sampling and (ii) shift the spectrum to the
**rest frame** and resample it onto one **common wavelength grid** (`REST_GRID`), so that thousands of
spectra can be stacked, compared and fed to PCA. Masked gaps stay `NaN` (never interpolated across).

Runs in parallel with `joblib`; results are cached on Drive (`CACHE_NPZ`, `CACHE_CSV`).

In [ ]:
def read_spectrum(path):
    # Read one .fits.gz spectrum -> dict {column: float numpy array}
    df_ = Table.read(path).to_pandas()          # masked values -> NaN
    return {c: df_[c].to_numpy(dtype=float) for c in [WAVE_COL] + FLUX_COLS + ERR_COLS}

def to_rest_grid(wave, flux, z, grid=REST_GRID, factor=1.5):
    # De-redshift (if needed) and linearly resample onto `grid`; pixels inside gaps -> NaN
    if WAVE_IS_OBSERVED:
        wave = wave / (1 + z)
    ok = np.isfinite(wave) & np.isfinite(flux)
    wave, flux = wave[ok], flux[ok]
    order = np.argsort(wave)
    wave, flux = wave[order], flux[order]
    if len(wave) < 10:
        return np.full(grid.shape, np.nan, np.float32)
    out = np.interp(grid, wave, flux, left=np.nan, right=np.nan)
    # Grid points whose two neighbouring native pixels are far apart lie inside a gap
    j = np.clip(np.searchsorted(wave, grid), 1, len(wave) - 1)
    out[(wave[j] - wave[j - 1]) > factor * np.median(np.diff(wave))] = np.nan
    return out.astype(np.float32)

def process_one(path, z):
    # Return (QC-statistics dict, {flux column: rest-frame resampled array}) for one file
    name = name_of(path)
    try:
        s = read_spectrum(path)
    except Exception as err:                    # corrupted / unreadable file
        return {'name': name, 'read_error': str(err)}, None
    w  = s[WAVE_COL]
    dw = np.diff(w)
    step = np.median(dw)
    row = {'name': name, 'read_error': None, 'n_pix': len(w),
           'wave_min': np.nanmin(w), 'wave_max': np.nanmax(w), 'step': step,
           'n_gaps': int((dw > 1.5 * step).sum()),
           'gap_frac': 1 - len(w) / ((np.nanmax(w) - np.nanmin(w)) / step + 1),  # missing pixels in range
           'is_sorted': bool(np.all(dw > 0))}
    for f, e in zip(FLUX_COLS, ERR_COLS):
        snr = np.where(s[e] > 0, s[f] / s[e], np.nan)
        row[f'med_{f}']     = np.nanmedian(s[f])
        row[f'snr_{f}']     = np.nanmedian(snr)
        row[f'negfrac_{f}'] = np.mean(s[f] < 0)
        row[f'nanfrac_{f}'] = np.mean(~np.isfinite(s[f]))
    z_ok = (not WAVE_IS_OBSERVED) or (np.isfinite(z) and z > -0.01)
    empty = np.full(REST_GRID.shape, np.nan, np.float32)
    rest = {f: to_rest_grid(w, s[f], z) if z_ok else empty for f in FLUX_COLS}
    return row, rest

In [ ]:
z_of  = meta.set_index('name')['nsa_z'].to_dict()     # redshift lookup by galaxy name
paths = spec_index['path'].tolist()[:N_MAX]            # [:None] -> all files

if CACHE_NPZ.exists() and CACHE_CSV.exists():
    # ---- Fast path: load the cache written by a previous run -----------------
    cache = np.load(CACHE_NPZ)
    assert np.allclose(cache['grid'], REST_GRID), 'REST_GRID changed: delete the cache files'
    FLUX_REST  = {f: cache[f] for f in FLUX_COLS}
    spec_stats = pd.read_csv(CACHE_CSV)
    print('Loaded cache:', FLUX_REST['FLUX_1RE'].shape)
else:
    # ---- Slow path: read every file in parallel (all CPU cores) --------------
    results = Parallel(n_jobs=-1, batch_size=32)(
        delayed(process_one)(p, z_of.get(name_of(p), np.nan)) for p in tqdm(paths, desc='Reading spectra'))
    spec_stats = pd.DataFrame([r[0] for r in results])
    empty = np.full(REST_GRID.shape, np.nan, np.float32)
    FLUX_REST = {f: np.vstack([r[1][f] if r[1] is not None else empty for r in results]) for f in FLUX_COLS}
    np.savez(CACHE_NPZ, grid=REST_GRID, **FLUX_REST)  # float32 arrays, ~0.2 GB per flux column for 16k
    spec_stats.to_csv(CACHE_CSV, index=False)
    print('Built and cached:', FLUX_REST['FLUX_1RE'].shape)

# One row per spectrum, aligned with the rows of FLUX_REST (left merge keeps the order)
df = spec_stats.merge(meta, on='name', how='left', validate='one_to_one')
print('Read errors:', df['read_error'].notna().sum())

In [ ]:
def normalise(F, window=NORM_WINDOW):
    # Divide each spectrum by its median flux in `window` -> shapes become comparable
    m = (REST_GRID >= window[0]) & (REST_GRID <= window[1])
    norm = np.nanmedian(F[:, m], axis=1)
    norm[~(norm > 0)] = np.nan
    return F / norm[:, None], norm

FN = {}                                               # normalised rest-frame spectra
for f in FLUX_COLS:
    FN[f], df[f'norm_{f}'] = normalise(FLUX_REST[f])

# Quality mask for spectra, and mask for galaxies usable in the classification analysis
df['good'] = df['read_error'].isna() & (df['snr_FLUX_1RE'] >= SNR_MIN) & np.isfinite(df['norm_FLUX_1RE'])
cls_mask   = (df['good'] & df['Ttype'].notna()).values
print(f"Good spectra: {df['good'].sum():,} / {len(df):,}  |  with Ttype: {cls_mask.sum():,}")

### 4.2 Quality-control EDA of the spectra

In [ ]:
qc_cols = ['n_pix', 'wave_min', 'wave_max', 'step', 'n_gaps', 'gap_frac']
fig, axes = plt.subplots(1, len(qc_cols), figsize=(20, 3.5))
for ax, c in zip(axes, qc_cols):
    df[c].plot.hist(bins=40, ax=ax, title=c)
plt.tight_layout(); plt.show()

print('Unsorted wavelength arrays:', (~df['is_sorted'].astype(bool)).sum())
df[qc_cols + [c for c in df.columns if c.startswith(('snr_', 'negfrac_', 'nanfrac_', 'med_'))]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Median S/N per spectrum for both apertures (log x-axis)
for f in FLUX_COLS:
    s = df[f'snr_{f}'].clip(lower=1e-3)
    axes[0].hist(np.log10(s.dropna()), bins=60, histtype='step', lw=2, label=f)
axes[0].axvline(np.log10(SNR_MIN), color='k', ls='--', label=f'S/N = {SNR_MIN}')
axes[0].set(xlabel='log10 median S/N', title='Signal-to-noise'); axes[0].legend()

# S/N vs mass and vs redshift, coloured by class
sns.scatterplot(data=df, x='LogMass', y='snr_FLUX_1RE', hue='early_late', s=5, alpha=.4, ax=axes[1])
sns.scatterplot(data=df, x='nsa_z',   y='snr_FLUX_1RE', hue='early_late', s=5, alpha=.4, ax=axes[2])
for ax in axes[1:]: ax.set_yscale('log')
axes[2].set_xscale('log')
plt.tight_layout(); plt.show()

### 4.3 Wavelength coverage of the stacked sample
Fraction of good spectra with valid flux at each rest wavelength: dips reveal commonly masked
regions (sky lines, detector issues), and the edges show the redshift-dependent coverage.

In [ ]:
# Main optical lines (vacuum wavelengths, Å) used for annotations
LINES = {'[OII]': 3728.5, 'CaK': 3934.8, 'CaH': 3969.6, 'Hδ': 4102.9, 'G': 4305.6, 'Hβ': 4862.7,
         '[OIII]': 5008.2, 'Mgb': 5176.7, 'NaD': 5895.6, 'Hα': 6564.6, '[NII]': 6585.3, '[SII]': 6725.0}

def mark_lines(ax, lines=LINES, xlim=None):
    # Draw dotted vertical lines + labels at the main spectral features
    lo, hi = xlim or ax.get_xlim()
    for lab, lam in lines.items():
        if lo < lam < hi:
            ax.axvline(lam, color='gray', ls=':', lw=.8)
            ax.text(lam, 1.0, lab, transform=ax.get_xaxis_transform(), rotation=90,
                    va='top', ha='right', fontsize=8, color='dimgray')

coverage = np.isfinite(FN['FLUX_1RE'][df['good'].values]).mean(axis=0)
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(REST_GRID, coverage)
ax.set(xlabel='Rest wavelength [Å]', ylabel='fraction valid', title='Coverage of the rest-frame grid')
mark_lines(ax); plt.show()

### 4.4 Random gallery of normalised rest-frame spectra

In [ ]:
idx = rng.choice(np.where(df['good'])[0], size=min(9, df['good'].sum()), replace=False)
fig, axes = plt.subplots(3, 3, figsize=(18, 9), sharex=True)
for ax, i in zip(axes.ravel(), idx):
    for f, c in zip(FLUX_COLS, ['tab:blue', 'tab:orange']):
        ax.plot(REST_GRID, FN[f][i], lw=.5, color=c, label=f)
    ax.set_title(f"{df.at[i, 'name']} | {df.at[i, 'Type']} | z={df.at[i, 'nsa_z']:.3f}", fontsize=9)
    ax.set_ylim(0, np.nanpercentile(FN['FLUX_1RE'][i], 99.5) * 1.3)
axes[0, 0].legend(); fig.supxlabel('Rest wavelength [Å]'); fig.supylabel('normalised flux')
plt.tight_layout(); plt.show()

### 4.5 `FLUX_1RE` vs `FLUX_FoV` (aperture effects)
The full field of view includes the outer disk; dividing FoV by 1Re spectra (both normalised) shows radial
colour/population gradients — typically bluer outskirts in spirals and flatter ratios in ellipticals.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Flux ratio (total light FoV / within 1Re), per galaxy
ratio = (df['med_FLUX_FoV'] / df['med_FLUX_1RE']).where(lambda s: s > 0)
sns.histplot(x=np.log10(ratio), hue=df['early_late'], element='step', stat='density',
             common_norm=False, bins=60, ax=axes[0])
axes[0].set(xlabel='log10 median(FLUX_FoV) / median(FLUX_1RE)', title='Aperture flux ratio')

# Median shape ratio FoV/1Re by morphological group
for g, c in zip(['E', 'S0/S0a', 'Spiral'], ['tab:red', 'tab:orange', 'tab:blue']):
    sel = cls_mask & (df['morph_group'] == g).values
    if sel.sum():
        axes[1].plot(REST_GRID, np.nanmedian(FN['FLUX_FoV'][sel] / FN['FLUX_1RE'][sel], axis=0),
                     color=c, label=f'{g} (N={sel.sum()})')
axes[1].axhline(1, color='k', lw=.5)
axes[1].set(xlabel='Rest wavelength [Å]', ylabel='FoV / 1Re (normalised)', title='Spectral shape ratio')
axes[1].set_ylim(0.6, 1.4); axes[1].legend()
plt.tight_layout(); plt.show()

## 5. Spectral indices
Compact, physically meaningful numbers measured on the rest-frame spectra (`FLUX_1RE`):

| Index | Definition | Physical meaning |
|---|---|---|
| `D4000n` | ⟨F⟩(4000–4100 Å) / ⟨F⟩(3850–3950 Å) | age of the stellar population (old → large) |
| `EW_Ha`, `EW_Hb`, `EW_OII`, `EW_OIII` | equivalent width, **positive = emission** | current star formation / nuclear activity |
| `Mgb`, `NaD` | equivalent width, **positive = absorption** | old, metal-rich populations |
| `color_BR` | −2.5 log10[⟨F⟩(4400–4600) / ⟨F⟩(6200–6400)] | continuum colour (red → large) |

$$\mathrm{EW} = \sum_\lambda \left(\frac{F_\lambda}{F_{c,\lambda}} - 1\right)\Delta\lambda$$
with the continuum $F_c$ linearly interpolated between a blue and a red side-band.
*Simplified definitions (F_λ, no velocity-dispersion or stellar-absorption corrections) — good for EDA, not for precision science.*

In [ ]:
STEP = REST_GRID[1] - REST_GRID[0]

def band_mean(F, lo, hi):
    # Mean flux of every spectrum inside [lo, hi] Å (NaN-aware)
    m = (REST_GRID >= lo) & (REST_GRID <= hi)
    return np.nanmean(F[:, m], axis=1)

def equivalent_width(F, line, blue, red, min_valid=0.6):
    # EW [Å] of a feature (positive = emission); NaN if < min_valid of the band is covered
    m  = (REST_GRID >= line[0]) & (REST_GRID <= line[1])
    cb, cr = band_mean(F, *blue), band_mean(F, *red)
    x  = REST_GRID[m]
    cont = cb[:, None] + (cr - cb)[:, None] * (x - np.mean(blue)) / (np.mean(red) - np.mean(blue))
    ew = np.nansum(F[:, m] / cont - 1, axis=1) * STEP
    enough = np.isfinite(F[:, m]).mean(axis=1) >= min_valid
    return np.where(enough & np.isfinite(cb) & np.isfinite(cr), ew, np.nan)

F = FN['FLUX_1RE']
df['D4000n']  = band_mean(F, 4000, 4100) / band_mean(F, 3850, 3950)
df['EW_Ha']   = equivalent_width(F, (6555, 6575), (6490, 6540), (6600, 6640))
df['EW_Hb']   = equivalent_width(F, (4852, 4872), (4815, 4845), (4880, 4910))
df['EW_OII']  = equivalent_width(F, (3716, 3740), (3690, 3714), (3742, 3766))
df['EW_OIII'] = equivalent_width(F, (4998, 5018), (4970, 4990), (5030, 5050))
df['Mgb']     = -equivalent_width(F, (5161.6, 5194.1), (5144.1, 5162.9), (5192.9, 5207.9))
df['NaD']     = -equivalent_width(F, (5878.5, 5911.0), (5862.2, 5877.3), (5923.7, 5949.7))
df['color_BR']= -2.5 * np.log10(band_mean(F, 4400, 4600) / band_mean(F, 6200, 6400))

INDEX_COLS = ['D4000n', 'EW_Ha', 'EW_Hb', 'EW_OII', 'EW_OIII', 'Mgb', 'NaD', 'color_BR']
df.loc[df['good'], INDEX_COLS].describe(percentiles=[.01, .5, .99]).T

## 6. Early- vs late-type galaxies: metadata, spectra and morphology

### 6.1 What to expect physically
| Property | **Early types** (E, S0; T ≤ 0) | **Late types** (spirals; T > 0) |
|---|---|---|
| Stellar population | old, metal-rich, little gas | mix of young + old stars, gas-rich |
| Continuum | red, strong **4000 Å break** (high D4000) | bluer, weaker break |
| Absorption lines | strong Ca II H&K, G band, **Mg b**, **Na D** | weaker metal lines, Balmer absorption from A stars |
| Emission lines | weak / none (sometimes LINER-like) | **Hα, [NII], Hβ, [OII], [OIII], [SII]** from HII regions |
| `LogMass` | higher on average | spans lower masses |
| `incl` (from axis ratio) | lower — round/thick systems rarely look "edge-on" | full range, up to ~90° |
| `PETRO_TH90`, `R90_kpc` | compact at fixed mass | more extended disks |
| FoV / 1Re spectral ratio | ~flat (weak gradients) | bluer outskirts (inside-out growth) |

Key caveat: **morphology correlates strongly with mass** (and, through MaNGA's selection, with redshift and aperture),
so some apparent "morphology" trends are really mass trends — §6.5 shows how to control for this.

### 6.2 Metadata vs class
Mann–Whitney U tests whether the early and late distributions differ (p-values are tiny for large samples;
look at the medians / effect size, not just p).

In [ ]:
cls = meta.dropna(subset=['early_late'])
rows = []
for col in PLOT_COLS:
    a = cls.loc[cls['early_late'] == 'Early', col].dropna()
    b = cls.loc[cls['early_late'] == 'Late',  col].dropna()
    U, p = stats.mannwhitneyu(a, b)
    rows.append({'variable': col, 'median_early': a.median(), 'median_late': b.median(),
                 'effect_size(AUC)': U / (len(a) * len(b)), 'p_value': p})
display(pd.DataFrame(rows).set_index('variable'))   # AUC: 0.5 = no difference, 0/1 = perfect separation

fig, axes = plt.subplots(1, len(PLOT_COLS), figsize=(20, 4))
for ax, col in zip(axes, PLOT_COLS):
    sns.violinplot(data=meta, x='morph_group', y=col, ax=ax, cut=0, inner='quartile', color='tab:cyan')
    ax.set_title(col)
axes[-1].set_yscale('log')
plt.tight_layout(); plt.show()

### 6.3 Spectral indices vs class
The **D4000 – EW(Hα)** plane is a classic diagnostic: early types cluster at high D4000 / no emission,
late types at low D4000 / strong Hα — a bimodality that mirrors the morphological one.

In [ ]:
good_cls = df[cls_mask]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, col in zip(axes.ravel(), INDEX_COLS):
    lo, hi = good_cls[col].quantile([.01, .99])          # clip extreme tails for readability
    sns.boxplot(data=good_cls[good_cls[col].between(lo, hi)], x='hubble_class', y=col,
                order=[h for h in HUBBLE_ORDER if h in good_cls['hubble_class'].unique()],
                ax=ax, color='lightsteelblue', fliersize=1)
    ax.tick_params(axis='x', rotation=60, labelsize=8); ax.set_xlabel('')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
kw = dict(c=good_cls['Ttype'].clip(-5, 10), cmap=HUBBLE_CMAP, vmin=-5, vmax=10, s=6, alpha=.6)

sc = axes[0].scatter(good_cls['D4000n'], good_cls['EW_Ha'], **kw)
axes[0].set(xlabel='D4000n', ylabel='EW(Hα) [Å] (emission > 0)', yscale='symlog', title='Age vs star formation')

axes[1].scatter(good_cls['LogMass'], good_cls['D4000n'], **kw)
axes[1].set(xlabel='LogMass', ylabel='D4000n', title='Mass vs age')

axes[2].scatter(good_cls['LogMass'], good_cls['color_BR'], **kw)
axes[2].set(xlabel='LogMass', ylabel='color_BR', title='Colour–mass diagram')
fig.colorbar(sc, ax=axes, label='Ttype (red = early, blue = late)')
plt.show()

### 6.4 Plotting spectra grouped by `Type` — suggestions

1. **Work in the rest frame and normalise** (here: median of 5400–5600 Å). Otherwise distance, mass and
   aperture dominate the y-axis and hide the shape differences.
2. **Stack instead of over-plotting**: median spectrum per group + 25–75 % band. Thousands of individual lines become unreadable.
3. **Order groups along the Hubble sequence** (median `Ttype`) and use **one consistent colormap**
   (red = early → blue = late) in every figure.
4. **Choose the grouping granularity**: `early_late` (2) → `morph_group` (3) → `hubble_class` (~12, bars merged) → `Type` (~25, with bars).
   Show `N` in the legend and skip groups with `N < MIN_N`.
5. **Offset ("waterfall") plots** to see each stack clearly along the sequence.
6. **Zoom panels** on diagnostic regions: 4000 Å break + Ca H&K, Hβ+[OIII], Mg b, Na D, Hα+[NII].
7. **2-D "spectral sequence" image**: individual spectra as rows sorted by `Ttype` — shows continuous trends and outliers at once.
8. **Ratio to a reference** (e.g. the global median): makes subtle differences (lines, slope) jump out.
9. **Control for mass** (and z): compare early vs late *at fixed `LogMass`* to separate morphology from mass effects.
10. Use `FLUX_1RE` for galaxy-to-galaxy comparisons (same physical aperture) and `FLUX_FoV`/`FLUX_1RE` for gradients.
11. Keep masked gaps as `NaN` — never interpolate across them.
12. **Show real examples too**: a small-multiples gallery of random galaxies per group to check the stacks are representative.

In [ ]:
FLUX  = 'FLUX_1RE'   # spectrum used in this section
MIN_N = 20           # minimum galaxies per group to show a stack

def ttype_color(t):
    # Consistent colour along the Hubble sequence: dark red (E, T=-5) -> dark blue (late spirals, T>=10)
    return HUBBLE_CMAP(np.clip((t + 5) / 15, 0, 1))

def stack_by(col, flux=FLUX, mask=None, min_n=MIN_N):
    # Median + interquartile range of normalised spectra per group of `col` (ordered by Ttype)
    mask = cls_mask if mask is None else mask
    order = df[mask].groupby(col, observed=True)['Ttype'].median().sort_values()
    out = {}
    for g, t in order.items():
        sel = mask & (df[col] == g).values
        if sel.sum() < min_n:
            continue
        Fg = FN[flux][sel]
        out[g] = dict(med=np.nanmedian(Fg, 0), lo=np.nanpercentile(Fg, 25, 0),
                      hi=np.nanpercentile(Fg, 75, 0), n=int(sel.sum()), ttype=t)
    return out

# ---- A) Early vs late, and the three morphological groups: median + IQR -------
fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for ax, col in zip(axes, ['early_late', 'morph_group']):
    for g, s in stack_by(col).items():
        c = ttype_color(s['ttype'])
        ax.plot(REST_GRID, s['med'], color=c, lw=1, label=f"{g} (N={s['n']})")
        ax.fill_between(REST_GRID, s['lo'], s['hi'], color=c, alpha=.25)
    ax.set(ylabel='normalised flux', title=f'Median {FLUX} stacks by {col}', ylim=(0, 2.2))
    ax.legend(); mark_lines(ax, xlim=(REST_GRID[0], REST_GRID[-1]))
axes[-1].set_xlabel('Rest wavelength [Å]')
plt.tight_layout(); plt.show()

In [ ]:
# ---- B) Waterfall: one stacked spectrum per Hubble class, offset vertically -----
GROUP_COL = 'hubble_class'   # try 'Type' to separate barred / unbarred
OFFSET = 0.6

stacks = stack_by(GROUP_COL)
fig, ax = plt.subplots(figsize=(14, 1.0 + 0.55 * len(stacks)))
for k, (g, s) in enumerate(stacks.items()):
    ax.plot(REST_GRID, s['med'] + k * OFFSET, color=ttype_color(s['ttype']), lw=.9)
    ax.text(REST_GRID[-1] + 30, 1 + k * OFFSET, f"{g}  (N={s['n']})", va='center', fontsize=9)
ax.set(xlabel='Rest wavelength [Å]', ylabel='normalised flux + offset',
       title=f'Spectral sequence: median {FLUX} by {GROUP_COL}', xlim=(REST_GRID[0], REST_GRID[-1] + 700))
mark_lines(ax, xlim=(REST_GRID[0], REST_GRID[-1]))
plt.show()

In [ ]:
# ---- C) Zoom panels on diagnostic features (stacks by Hubble class) -------------
WINDOWS = {'D4000 + Ca H&K': (3700, 4200), 'Hβ + [OIII]': (4800, 5050),
           'Mg b': (5100, 5250), 'Na D': (5840, 5960), 'Hα + [NII] + [SII]': (6500, 6760)}

fig, axes = plt.subplots(1, len(WINDOWS), figsize=(22, 4.5))
for ax, (title, (lo, hi)) in zip(axes, WINDOWS.items()):
    m = (REST_GRID >= lo) & (REST_GRID <= hi)
    for g, s in stacks.items():
        ax.plot(REST_GRID[m], s['med'][m], color=ttype_color(s['ttype']), lw=1, label=g)
    ax.set(title=title, xlim=(lo, hi)); mark_lines(ax, xlim=(lo, hi))
axes[0].set_ylabel('normalised flux'); axes[-1].legend(fontsize=7, ncol=2)
fig.supxlabel('Rest wavelength [Å]'); plt.tight_layout(); plt.show()

In [ ]:
# ---- D) 2-D spectral sequence: individual spectra sorted by Ttype ---------------
N_ROWS = 2000
rows_idx = np.where(cls_mask)[0]
rows_idx = rng.choice(rows_idx, size=min(N_ROWS, len(rows_idx)), replace=False)
rows_idx = rows_idx[np.argsort(df['Ttype'].values[rows_idx], kind='stable')]   # sort by Ttype

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(FN[FLUX][rows_idx], aspect='auto', cmap='magma', vmin=0.2, vmax=1.6,
               extent=[REST_GRID[0], REST_GRID[-1], len(rows_idx), 0], interpolation='nearest')
# Horizontal separators + labels at Ttype boundaries
tt = df['Ttype'].values[rows_idx]
for t in np.unique(tt):
    first, last = np.argmax(tt == t), len(tt) - np.argmax(tt[::-1] == t)
    ax.axhline(first, color='w', lw=.4)
    ax.text(REST_GRID[0] - 40, (first + last) / 2, f'T={t:g}', ha='right', va='center', fontsize=8)
fig.colorbar(im, label='normalised flux')
ax.set(xlabel='Rest wavelength [Å]', yticks=[], title=f'{len(rows_idx)} spectra sorted by Ttype (top = early)')
plt.show()

In [ ]:
# ---- E) Ratio of each stack to the global median: highlights subtle differences --
global_med = np.nanmedian(FN[FLUX][cls_mask], axis=0)
fig, ax = plt.subplots(figsize=(15, 4.5))
for g, s in stacks.items():
    ax.plot(REST_GRID, s['med'] / global_med, color=ttype_color(s['ttype']), lw=.8, label=g)
ax.axhline(1, color='k', lw=.5)
ax.set(xlabel='Rest wavelength [Å]', ylabel='stack / global median', ylim=(0.5, 1.6),
       title='Relative spectral differences along the Hubble sequence')
ax.legend(ncol=6, fontsize=8); mark_lines(ax, xlim=(REST_GRID[0], REST_GRID[-1])); plt.show()

In [ ]:
# ---- F) Small-multiples gallery: random real spectra per morphological group ----
N_EX = 4
groups = ['E', 'S0/S0a', 'Spiral']
fig, axes = plt.subplots(len(groups), N_EX, figsize=(20, 9), sharex=True, sharey=True)
for r, g in enumerate(groups):
    pool = np.where(cls_mask & (df['morph_group'] == g).values)[0]
    for c, i in enumerate(rng.choice(pool, size=min(N_EX, len(pool)), replace=False)):
        ax = axes[r, c]
        ax.plot(REST_GRID, FN[FLUX][i], lw=.4, color=ttype_color(df.at[i, 'Ttype']))
        ax.set_title(f"{df.at[i, 'name']} | {df.at[i, 'Type']} | logM={df.at[i, 'LogMass']:.1f}", fontsize=8)
    axes[r, 0].set_ylabel(g)
axes[0, 0].set_ylim(0, 2.2); fig.supxlabel('Rest wavelength [Å]')
plt.tight_layout(); plt.show()

### 6.5 Controlling for stellar mass
If early- and late-type stacks still differ **at the same `LogMass`**, the difference is linked to morphology
(star-formation history, gas content), not just to mass.

In [ ]:
MASS_BINS = [(9.5, 10.0), (10.0, 10.5), (10.5, 11.0), (11.0, 11.5)]
fig, axes = plt.subplots(1, len(MASS_BINS), figsize=(22, 4.5), sharey=True)
for ax, (m0, m1) in zip(axes, MASS_BINS):
    in_bin = cls_mask & df['LogMass'].between(m0, m1).values
    for g, s in stack_by('early_late', mask=in_bin, min_n=10).items():
        c = ttype_color(s['ttype'])
        ax.plot(REST_GRID, s['med'], color=c, lw=.8, label=f"{g} (N={s['n']})")
        ax.fill_between(REST_GRID, s['lo'], s['hi'], color=c, alpha=.2)
    ax.set(title=f'{m0} ≤ LogMass < {m1}', ylim=(0, 2.2)); ax.legend(fontsize=8)
axes[0].set_ylabel('normalised flux'); fig.supxlabel('Rest wavelength [Å]')
plt.tight_layout(); plt.show()

### 6.6 PCA of the spectra
PCA compresses each ~3,000-pixel spectrum into a few numbers. If PC1/PC2 separate the colours (Ttype),
the spectra alone carry most of the morphological information.

In [ ]:
from sklearn.decomposition import PCA

# Keep only wavelengths present in >=95% of good spectra; fill remaining NaNs with the pixel median
wmask = coverage >= 0.95
X = FN[FLUX][cls_mask][:, wmask]
X = np.where(np.isfinite(X), X, np.nanmedian(X, axis=0))

pca = PCA(n_components=10, random_state=SEED).fit(X)
PCS = pca.transform(X)
print('Explained variance (first 5):', np.round(pca.explained_variance_ratio_[:5], 3))

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
sc = axes[0].scatter(PCS[:, 0], PCS[:, 1], c=df.loc[cls_mask, 'Ttype'].clip(-5, 10),
                     cmap=HUBBLE_CMAP, vmin=-5, vmax=10, s=4, alpha=.6)
axes[0].set(xlabel='PC1', ylabel='PC2', title='Spectra in PCA space'); fig.colorbar(sc, ax=axes[0], label='Ttype')
for k in range(3):                                    # eigen-spectra
    axes[1].plot(REST_GRID[wmask], pca.components_[k] + 0.1 * k, lw=.7, label=f'PC{k+1}')
axes[1].set(xlabel='Rest wavelength [Å]', title='Eigen-spectra (offset)'); axes[1].legend()
plt.tight_layout(); plt.show()

### 6.7 How much does each data source tell us about early vs late type?
A quick logistic-regression baseline (5-fold cross-validated ROC-AUC). It's an EDA diagnostic, not a final classifier:
it shows how much early/late information each feature set carries on its own.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score

y = (df.loc[cls_mask, 'early_late'] == 'Early').astype(int).values
META_FEATS = ['LogMass', 'incl', 'R90_kpc', 'nsa_z']
feature_sets = {
    'metadata':        df.loc[cls_mask, META_FEATS].values,
    'spectral indices': df.loc[cls_mask, INDEX_COLS].values,
    'spectral PCA(10)': PCS,
}
feature_sets['all combined'] = np.hstack(list(feature_sets.values()))

model = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(max_iter=2000))
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
auc = {k: cross_val_score(model, Xf, y, cv=cv, scoring='roc_auc') for k, Xf in feature_sets.items()}
display(pd.DataFrame({k: [v.mean(), v.std()] for k, v in auc.items()}, index=['AUC mean', 'AUC std']).T)

# Standardised coefficients: which metadata/index features push towards "Early"?
names = META_FEATS + INDEX_COLS
model.fit(np.hstack([feature_sets['metadata'], feature_sets['spectral indices']]), y)
coef = pd.Series(model[-1].coef_[0], index=names).sort_values()
coef.plot.barh(color=np.where(coef > 0, 'tab:red', 'tab:blue'), figsize=(7, 5),
               title='Logistic coefficients (>0 = more likely Early)'); plt.show()

### 6.8 Optional: images with Marvin & next steps with astroNN
**Marvin** (SDSS MaNGA toolkit) can download the optical image of any galaxy by `plateifu`
(`manga-10001-1901` → `10001-1901`) — useful to visually check the morphology behind outliers
(e.g. "early types" with strong Hα, or spirals with high D4000). Requires internet access to the SDSS servers.

In [ ]:
# !pip install -q sdss-marvin        # uncomment to install (may take a minute)
try:
    from marvin import config
    from marvin.tools.image import Image
    config.setRelease('DR17')         # public MaNGA data release
    config.mode = 'remote'

    # Example: 'early' galaxies with the strongest Hα emission (possible misclassifications / AGN)
    odd = df[cls_mask & (df['early_late'] == 'Early').values].nlargest(4, 'EW_Ha')
    fig, axes = plt.subplots(1, len(odd), figsize=(16, 4))
    for ax, (_, r) in zip(axes, odd.iterrows()):
        img = Image(plateifu=r['name'].replace('manga-', ''))
        ax.imshow(img.data); ax.axis('off')
        ax.set_title(f"{r['name']}\n{r['Type']}  EW(Hα)={r['EW_Ha']:.1f}", fontsize=9)
    plt.show()
except Exception as err:
    print('Marvin not available or offline -> skipped:', err)

**Suggested next steps**
* **astroNN** (`pip install astroNN`): build a 1-D neural network on the normalised rest-frame arrays `FN['FLUX_1RE']`
  (already on a common grid) to predict `Ttype`, or use its Galaxy10 CNN tools on Marvin images and compare
  image-based and spectrum-based classifications.
* Fit stellar populations / emission lines properly (e.g. pPXF) and build a BPT diagram to separate star formation from AGN/LINER emission in early types.
* Extend §6.5 by matching early/late samples in mass **and** redshift.

### 6.9 Save the EDA table
One row per galaxy: metadata + QC statistics + spectral indices, ready for modelling.

In [ ]:
OUT_CSV = DRIVE_DIR / 'eda_galaxy_table.csv'
df.to_csv(OUT_CSV, index=False)
print('Saved', OUT_CSV, df.shape)